In [ ]:
# CELL 1
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# CELL 2
import os, sys
PROJECT_DIR = '/content/drive/MyDrive/TDA_Project'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for d in ['small_gpt_web/attentions', 'small_gpt_web/barcodes', 'small_gpt_web/features']:
    os.makedirs(d, exist_ok=True)
print("Listo:", os.getcwd())


Listo: /content/drive/MyDrive/TDA_Project


In [ ]:
# CELL 3
!pip install -q ripser persim kmapper networkx transformers torch tqdm scikit-learn


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 15.7 MB/s eta 0:00:00


In [ ]:


from collections import defaultdict
import itertools
import re
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import BertTokenizer, BertModel, BertForSequenceClassification

from stats_count import *
from grab_weights import grab_attention_weights, text_preprocessing

In [ ]:


import warnings

warnings.filterwarnings('ignore')

In [ ]:


get_ipython().system('env | grep CUDA_VISIBLE')


# ## Parameters

In [ ]:


np.random.seed(42) # For reproducibility.

In [ ]:
max_tokens_amount  = 128 # The number of tokens to which the tokenized text is truncated / padded.

layers_of_interest = [i for i in range(12)]  # Layers for which attention matrices and features on them are
                                             # calculated. For calculating features on all layers, leave it be
                                             # [i for i in range(12)].

model_path = tokenizer_path = "fine_tuned_bert"

# You can use either standard or fine-tuned BERT. If you want to use fine-tuned BERT to your current task, save the
# model and the tokenizer with the commands tokenizer.save_pretrained(output_dir);
# bert_classifier.save_pretrained(output_dir) into the same directory and insert the path to it here.

In [ ]:


subset = "test_5k"           # .csv file with the texts, for which we count topological features
input_dir = "small_gpt_web/"  # Name of the directory with .csv file
output_dir = "small_gpt_web/" # Name of the directory with calculations results

prefix = output_dir + subset

r_file     = output_dir + 'attentions/' + subset  + "_all_heads_" + str(len(layers_of_interest)) + "_layers_MAX_LEN_" + \
             str(max_tokens_amount) + "_" + model_path.split("/")[-1]
# Name of the file for attention matrices weights

barcodes_file = output_dir + 'barcodes/' + subset  + "_all_heads_" + str(len(layers_of_interest)) + "_layers_MAX_LEN_" + \
             str(max_tokens_amount) + "_" + model_path.split("/")[-1]
# Name of the file for barcodes information

In [ ]:


r_file

'small_gpt_web/attentions/test_5k_all_heads_12_layers_MAX_LEN_128_fine_tuned_bert'

In [ ]:


barcodes_file


# .csv file must contain the column with the name **sentence** with the texts. It can also contain the column **labels**, which will be needed for testing. Any other arbitrary columns will be ignored.

'small_gpt_web/barcodes/test_5k_all_heads_12_layers_MAX_LEN_128_fine_tuned_bert'

In [ ]:


try:
    data = pd.read_csv(input_dir + subset + ".csv").reset_index(drop=True)
except:
    #data = pd.read_csv(input_dir + subset + ".tsv", delimiter="\t")
    data = pd.read_csv(input_dir + subset + ".tsv", delimiter="\t", header=None)
    data.columns = ["0", "labels", "2", "sentence"]

# Truncate dataset to 500 records for speed
data = data.iloc[:500]
print(f'Truncated data to {len(data)} records')

Truncated data to 500 records


In [ ]:


data.head()

,Unnamed: 0,id,ended,length,sentence,label
0,4722,259722,True,231,The Learning Co.\n\nDeveloped by\n\nThe Learni...,natural
1,2757,257813,True,563,Bush doubles down on foreign policy on Saturda...,generated
2,2194,257194,True,62,Here are six interesting things you need to kn...,natural
3,817,255817,True,293,Introduction\n\nWe would like to thank Antec f...,natural
4,3886,258886,False,1024,"ELKRIDGE, Md.—A group called ""Muslims for Trum...",natural


In [ ]:


sentences = data['sentence']
print("Average amount of words in example:", \
      np.mean(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Max. amount of words in example:", \
      np.max(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Min. amount of words in example:", \
      np.min(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))

Average amount of words in example: 2706.578
Max. amount of words in example: 5229
Min. amount of words in example: 34


In [ ]:
def get_token_length(batch_texts):
    inputs = tokenizer(list(batch_texts),
       return_tensors='pt',
       add_special_tokens=True,
       max_length=MAX_LEN,             # Max length to truncate/pad
       padding='max_length',           # Pad sentence to max length
       truncation=True
    )
    inputs = inputs['input_ids'].numpy()
    n_tokens = []
    indexes = np.argwhere(inputs == tokenizer.pad_token_id)
    for i in range(inputs.shape[0]):
        ids = indexes[(indexes == i)[:, 0]]
        if not len(ids):
            n_tokens.append(MAX_LEN)
        else:
            n_tokens.append(ids[0, 1])
    return n_tokens

In [ ]:


MAX_LEN = max_tokens_amount
tokenizer = BertTokenizer.from_pretrained(tokenizer_path, do_lower_case=True)

In [ ]:
data['tokenizer_length'] = get_token_length(data['sentence'].values)

In [ ]:


data

,Unnamed: 0,id,ended,length,sentence,label,tokenizer_length
0,4722,259722,True,231,The Learning Co.\n\nDeveloped by\n\nThe Learni...,natural,128
1,2757,257813,True,563,Bush doubles down on foreign policy on Saturda...,generated,128
2,2194,257194,True,62,Here are six interesting things you need to kn...,natural,71
3,817,255817,True,293,Introduction\n\nWe would like to thank Antec f...,natural,128
4,3886,258886,False,1024,"ELKRIDGE, Md.—A group called ""Muslims for Trum...",natural,128
...,...,...,...,...,...,...,...
495,4035,259119,True,907,Trophies : A 1968 Kentucky steel plate set is ...,generated,128
496,3199,258199,True,742,Researchers from Italy and Portugal announced ...,natural,128
497,2475,257475,True,229,$6 .20 This item ships free (worldwide) This i...,natural,128
498,1372,256401,True,554,What are risk management processes for an open...,generated,128


In [ ]:


ntokens_array = data['tokenizer_length'].values

In [ ]:


from math import ceil

batch_size = 10 # batch size
number_of_batches = ceil(len(data['sentence']) / batch_size)
DUMP_SIZE = 100 # number of batches to be dumped


# ## Calculating Ripser features

# Format: "h{dim}\_{type}\_{args}"
#
# Dimension: 0, 1, etc.; homology dimension
#
# Types:
#
#     1. s: sum of lengths; example: "h1_s".
#     2. m: mean of lengths; example: "h1_m"
#     3. v: variance of lengths; example "h1_v"
#     4. n: number of barcodes with time of birth/death more/less then threshold.
#         4.1. b/d: birth or death
#         4.2. m/l: more or less than threshold
#         4.2. t: threshold value
#        example: "h0_n_d_m_t0.5", "h1_n_b_l_t0.75"
#     5. t: time of birth/death of the longest barcode (not incl. inf).
#         3.1. b/d: birth of death
#         example: "h0_t_d", "h1_t_b"
#     6. nb: number of barcodes in dim
#        example: h0_nb
#     7. e: entropy; example: "h1_e"

In [ ]:


import os
import timeit
import ripser_count

adj_filenames = [
    output_dir + 'attentions/' + filename
    for filename in os.listdir(output_dir + 'attentions/') if r_file in (output_dir + 'attentions/' + filename)
]
# sorted by part number
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip()))
adj_filenames

['small_gpt_web/attentions/test_5k_all_heads_12_layers_MAX_LEN_128_fine_tuned_bert_part1of1.npy']

In [ ]:


dim = 1
lower_bound = 1e-3


# ## Calculating and saving barcodes

In [ ]:


from multiprocessing import Process, Queue

def subprocess_wrap(queue, function, args):
    queue.put(function(*args))
#     print("Putted in Queue")
    queue.close()
    exit()

In [ ]:


import json
import itertools
from collections import defaultdict

def get_only_barcodes(adj_matricies, ntokens_array, dim, lower_bound):
    """Get barcodes from adj matricies for each layer, head"""
    barcodes = {}
    layers, heads = range(adj_matricies.shape[1]), range(adj_matricies.shape[2])
    for (layer, head) in itertools.product(layers, heads):
        matricies = adj_matricies[:, layer, head, :, :]
        barcodes[(layer, head)] = ripser_count.get_barcodes(matricies, ntokens_array, dim, lower_bound, (layer, head))
    return barcodes

def format_barcodes(barcodes):
    """Reformat barcodes to json-compatible format"""
    return [{d: b[d].tolist() for d in b} for b in barcodes]

def save_barcodes(barcodes, filename):
    """Save barcodes to file"""
    formatted_barcodes = defaultdict(dict)
    for layer, head in barcodes:
        formatted_barcodes[layer][head] = format_barcodes(barcodes[(layer, head)])
    json.dump(formatted_barcodes, open(filename, 'w'))

def unite_barcodes(barcodes, barcodes_part):
    """Unite 2 barcodes"""
    for (layer, head) in barcodes_part:
        barcodes[(layer, head)].extend(barcodes_part[(layer, head)])
    return barcodes

def split_matricies_and_lengths(adj_matricies, ntokens, number_of_splits):
    splitted_ids = np.array_split(np.arange(ntokens.shape[0]), number_of_splits)
    splitted = [(adj_matricies[ids], ntokens[ids]) for ids in splitted_ids]
    return splitted

In [ ]:
number_of_splits = 1
for i, filename in enumerate(tqdm(adj_filenames, desc='Calculating barcodes')):
    barcodes = defaultdict(list)
    adj_matricies = np.load(filename, allow_pickle=True) # samples X
    print(f"Matricies loaded from: {filename}")
    ntokens = ntokens_array[i*batch_size*DUMP_SIZE : (i+1)*batch_size*DUMP_SIZE]

    # Ensure ntokens length matches adj_matricies length to prevent IndexError
    ntokens = ntokens[:adj_matricies.shape[0]]

    splitted = split_matricies_and_lengths(adj_matricies, ntokens, number_of_splits)

    for matricies, ntokens in tqdm(splitted, leave=False):
        # Run sequentially! No multiprocessing to bypass MacOS spawn errors.
        barcodes_part = get_only_barcodes(matricies, ntokens, dim, lower_bound)
        barcodes = unite_barcodes(barcodes, barcodes_part)

    part = filename.split('_')[-1].split('.')[0]
    save_barcodes(barcodes, barcodes_file + '_' + part + '.json')

Calculating barcodes:   0%|          | 0/1 [00:00<?, ?it/s]

Matricies loaded from: small_gpt_web/attentions/test_5k_all_heads_12_layers_MAX_LEN_128_fine_tuned_bert_part1of1.npy


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:


ripser_feature_names=[
    'h0_s',
    'h0_e',
    'h0_t_d',
    'h0_n_d_m_t0.75',
    'h0_n_d_m_t0.5',
    'h0_n_d_l_t0.25',
    'h1_t_b',
    'h1_n_b_m_t0.25',
    'h1_n_b_l_t0.95',
    'h1_n_b_l_t0.70',
    'h1_s',
    'h1_e',
    'h1_v',
    'h1_nb'
]

In [ ]:


import os
import timeit
import ripser_count
import json

adj_filenames = [
    output_dir + 'barcodes/' + filename
    for filename in os.listdir(output_dir + 'barcodes/') if r_file.split('/')[-1] == filename.split('_part')[0]
]
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip()))
adj_filenames

In [ ]:


def reformat_barcodes(barcodes):
    """Return barcodes to their original format"""
    formatted_barcodes = []
    for barcode in barcodes:
        formatted_barcode = {}
        for dim in barcode:
            formatted_barcode[int(dim)] = np.asarray(
                [(b, d) for b,d in barcode[dim]], dtype=[('birth', '<f4'), ('death', '<f4')]
            )
        formatted_barcodes.append(formatted_barcode)
    return formatted_barcodes

In [ ]:


features_array = []

for filename in tqdm(adj_filenames, desc='Calculating ripser++ features'):
    barcodes = json.load(open(filename))
    print(f"Barcodes loaded from: {filename}", flush=True)
    features_part = []
    for layer in barcodes:
        features_layer = []
        for head in barcodes[layer]:
            ref_barcodes = reformat_barcodes(barcodes[layer][head])
            features = ripser_count.count_ripser_features(ref_barcodes, ripser_feature_names)
            features_layer.append(features)
        features_part.append(features_layer)
    features_array.append(np.asarray(features_part))

In [ ]:


ripser_file = output_dir + 'features/' + subset + "_all_heads_" + str(len(layers_of_interest)) + "_layers" \
             + "_MAX_LEN_" + str(max_tokens_amount) + \
             "_" + model_path.split("/")[-1] + "_ripser" + '.npy'
ripser_file

In [ ]:


features = np.concatenate(features_array, axis=2)
features.shape

In [ ]:
from transformers import BertModel, BertTokenizer
from grab_weights import text_preprocessing
import torch
import numpy as np
from tqdm import tqdm

def grab_attention_weights(model, tokenizer, sentences, MAX_LEN, device='cuda:0'):
    all_attentions = []
    batch_size = 10
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), batch_size), desc='Extracting Attentions'):
            batch_texts = [text_preprocessing(s) for s in sentences[i:i+batch_size]]
            inputs = tokenizer(
                batch_texts,
                return_tensors='pt',
                add_special_tokens=True,
                max_length=MAX_LEN,
                padding='max_length',
                truncation=True
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_attentions=True)
            # outputs.attentions is a tuple of 12 layers: each (batch, heads, seq_len, seq_len)
            # We stack them to shape (batch, layers, heads, seq_len, seq_len)
            attentions = torch.stack(outputs.attentions, dim=1)
            all_attentions.append(attentions.cpu().numpy())
    return np.concatenate(all_attentions, axis=0)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f"Loading model and tokenizer from {model_path}...")
tokenizer_obj = BertTokenizer.from_pretrained(tokenizer_path, do_lower_case=True)
model_obj = BertModel.from_pretrained(model_path, output_attentions=True)
model_obj = model_obj.to(device)

print("Extracting attention weights...")
attention_matrices = grab_attention_weights(
    model_obj,
    tokenizer_obj,
    data['sentence'].tolist(),
    max_tokens_amount,
    device=device
)

# Save the output to disk so the barcode script can load it
out_file = r_file + "_part1of1.npy"
print(f"\nSaving attention matrices to {out_file}...")
np.save(out_file, attention_matrices)

print("Extraction complete. You can now run the barcode and features calculation cells.")

In [ ]:


np.save(ripser_file, features)


# ## Calculating template features

In [ ]:


import os
from multiprocessing import Pool

In [ ]:
tokenizer = BertTokenizer.from_pretrained(tokenizer_path, do_lower_case=True)

In [ ]:


def matrix_distance(matricies, template, broadcast=True):
    """
    Calculates the distance between the list of matricies and the template matrix.
    Args:

    -- matricies: np.array of shape (n_matricies, dim, dim)
    -- template: np.array of shape (dim, dim) if broadcast else (n_matricies, dim, dim)

    Returns:
    -- diff: np.array of shape (n_matricies, )
    """
    diff = np.linalg.norm(matricies-template, ord='fro', axis=(1, 2))
    div = np.linalg.norm(matricies, ord='fro', axis=(1, 2))**2
    if broadcast:
        div += np.linalg.norm(template, ord='fro')**2
    else:
        div += np.linalg.norm(template, ord='fro', axis=(1, 2))**2
    return diff/np.sqrt(div)

In [ ]:
attention_dir = output_dir + 'attentions/'
attention_name = r_file.split('/')[-1] # Dynamically gets the current model's attention file prefix

texts_name = input_dir + subset + '.csv'

MAX_LEN = max_tokens_amount
print(f"Using attention files starting with: {attention_name}")

In [ ]:


def attention_to_self(matricies):
    """
    Calculates the distance between input matricies and identity matrix,
    which representes the attention to the same token.
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.eye(n)
    return matrix_distance(matricies, template_matrix)

def attention_to_next_token(matricies):
    """
    Calculates the distance between input and E=(i, i+1) matrix,
    which representes the attention to the next token.
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.triu(np.tri(n, k=1, dtype=matricies.dtype), k=1)
    return matrix_distance(matricies, template_matrix)

def attention_to_prev_token(matricies):
    """
    Calculates the distance between input and E=(i+1, i) matrix,
    which representes the attention to the previous token.
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.triu(np.tri(n, k=-1, dtype=matricies.dtype), k=-1)
    return matrix_distance(matricies, template_matrix)

def attention_to_beginning(matricies):
    """
    Calculates the distance between input and E=(i+1, i) matrix,
    which representes the attention to [CLS] token (beginning).
    """
    _, n, m = matricies.shape
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
    template_matrix = np.zeros((n, n))
    template_matrix[:, 0] = 1.0
    return matrix_distance(matricies, template_matrix)

def attention_to_ids(matricies, list_of_ids, token_id):
    """
    Calculates the distance between input and ids matrix,
    which representes the attention to some particular tokens,
    which ids are in the `list_of_ids` (commas, periods, separators).
    """

    batch_size, n, m = matricies.shape
    EPS = 1e-7
    assert n == m, f"Input matrix has shape {n} x {m}, but the square matrix is expected"
#     assert len(list_of_ids) == batch_size, f"List of ids length doesn't match the dimension of the matrix"
    template_matrix = np.zeros_like(matricies)
    ids = np.argwhere(list_of_ids == token_id)
    if len(ids):
        batch_ids, row_ids = zip(*ids)
        template_matrix[np.array(batch_ids), :, np.array(row_ids)] = 1.0
        template_matrix /= (np.sum(template_matrix, axis=-1, keepdims=True) + EPS)
    return matrix_distance(matricies, template_matrix, broadcast=False)

In [ ]:
# Replaced local definition with module import to fix multiprocessing AttributeError
from template_features import matrix_distance, attention_to_self, attention_to_next_token, attention_to_prev_token, attention_to_beginning, attention_to_ids, count_template_features, calculate_features_t

In [ ]:
%%writefile template_features.py
import numpy as np

def matrix_distance(matricies, template, broadcast=True):
    diff = np.linalg.norm(matricies-template, ord='fro', axis=(1, 2))
    div = np.linalg.norm(matricies, ord='fro', axis=(1, 2))**2
    if broadcast:
        div += np.linalg.norm(template, ord='fro')**2
    else:
        div += np.linalg.norm(template, ord='fro', axis=(1, 2))**2
    return diff/np.sqrt(div)

def attention_to_self(matricies):
    _, n, m = matricies.shape
    template_matrix = np.eye(n)
    return matrix_distance(matricies, template_matrix)

def attention_to_next_token(matricies):
    _, n, m = matricies.shape
    template_matrix = np.triu(np.tri(n, k=1, dtype=matricies.dtype), k=1)
    return matrix_distance(matricies, template_matrix)

def attention_to_prev_token(matricies):
    _, n, m = matricies.shape
    template_matrix = np.triu(np.tri(n, k=-1, dtype=matricies.dtype), k=-1)
    return matrix_distance(matricies, template_matrix)

def attention_to_beginning(matricies):
    _, n, m = matricies.shape
    template_matrix = np.zeros((n, n))
    template_matrix[:, 0] = 1.0
    return matrix_distance(matricies, template_matrix)

def attention_to_ids(matricies, list_of_ids, token_id):
    batch_size, n, m = matricies.shape
    EPS = 1e-7
    template_matrix = np.zeros_like(matricies)
    ids = np.argwhere(list_of_ids == token_id)
    if len(ids):
        batch_ids, row_ids = zip(*ids)
        template_matrix[np.array(batch_ids), :, np.array(row_ids)] = 1.0
        template_matrix /= (np.sum(template_matrix, axis=-1, keepdims=True) + EPS)
    return matrix_distance(matricies, template_matrix, broadcast=False)

def count_template_features(matricies, feature_list, list_of_ids):
    features = []
    if 'self' in feature_list:
        features.append(attention_to_self(matricies))
    if 'beginning' in feature_list:
        features.append(attention_to_beginning(matricies))
    if 'prev' in feature_list:
        features.append(attention_to_prev_token(matricies))
    if 'next' in feature_list:
        features.append(attention_to_next_token(matricies))
    if 'comma' in feature_list:
        features.append(attention_to_ids(matricies, list_of_ids, 1010))
    if 'dot' in feature_list:
        features.append(attention_to_ids(matricies, list_of_ids, 1012))
    return np.array(features).T

def calculate_features_t(matricies, feature_list, list_of_ids):
    batch, layers, heads, n, m = matricies.shape
    res = []
    for l in range(layers):
        res_l = []
        for h in range(heads):
            m_lh = matricies[:, l, h, :, :]
            features = count_template_features(m_lh, feature_list, list_of_ids)
            res_l.append(features)
        res.append(res_l)
    # Transpose back to match the notebook's expected shape
    return np.array(res).transpose((2, 0, 1, 3))

In [ ]:
# '.' id 1012
# ',' id 1010
def get_list_of_ids(sentences, tokenizer):
    inputs = tokenizer([text_preprocessing(s) for s in sentences],
                                       add_special_tokens=True,
                                       max_length=MAX_LEN,             # Max length to truncate/pad
                                       padding='max_length',         # Pad sentence to max length)
                                       truncation=True
                                      )
    return np.array(inputs['input_ids'])

In [ ]:


num_of_workers = 20
pool = Pool(num_of_workers)
feature_list = ['self', 'beginning', 'prev', 'next', 'comma', 'dot']

In [ ]:


adj_filenames = [
    attention_dir + filename
    for filename in os.listdir(attention_dir)
    if attention_name == filename.split("_part")[0]
]
# sorted by part number
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip()))
adj_filenames

In [ ]:


attention_name

In [ ]:


texts = pd.read_csv(texts_name)

In [ ]:


features_array = []

for i, filename in tqdm(list(enumerate(adj_filenames)), desc='Features calc'):
    adj_matricies = np.load(filename, allow_pickle=True)
    batch_size = adj_matricies.shape[0]
    sentences = texts['sentence'].values[i*batch_size:(i+1)*batch_size]
    splitted_indexes = np.array_split(np.arange(batch_size), num_of_workers)
    splitted_list_of_ids = [
        get_list_of_ids(sentences[indx], tokenizer)
        for indx in tqdm(splitted_indexes, desc=f"Calculating token ids on iter {i} from {len(adj_filenames)}")
    ]
    splitted_adj_matricies = [adj_matricies[indx] for indx in splitted_indexes]

    args = [(m, feature_list, list_of_ids) for m, list_of_ids in zip(splitted_adj_matricies, splitted_list_of_ids)]

    features_array_part = pool.starmap(
        calculate_features_t, args
    )
    features_array.append(np.concatenate([_ for _ in features_array_part], axis=3))
features_array = np.concatenate(features_array, axis=3)

In [ ]:


"small_gpt_web/features/" + attention_name + "_template.npy"

In [ ]:


np.save("small_gpt_web/features/" + attention_name + "_template.npy", features_array)

In [ ]:





# ## Plotting TDA Features (H0, H1) and Bottleneck Distances
#
# The following cells will load the computed features and barcodes, then generate boxplots and average persistence diagrams.

In [ ]:


import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import persim
import glob
import json
import os

def plot_feature_boxplots(labels, features, feature_names):
    print("Plotting feature boxplots...")
    # features shape: (layers, heads, samples, num_features)
    # average across layers and heads for macroscopic view
    features_avg = np.mean(features, axis=(0, 1))

    nat_mask = labels == 'natural'
    gen_mask = labels == 'generated'

    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    axes = axes.flatten()

    for i, name in enumerate(feature_names):
        nat_data = features_avg[nat_mask, i]
        gen_data = features_avg[gen_mask, i]
        # remove nans
        nat_data = nat_data[~np.isnan(nat_data)]
        gen_data = gen_data[~np.isnan(gen_data)]
        if len(nat_data) > 0 and len(gen_data) > 0:
            axes[i].boxplot([nat_data, gen_data], labels=['Natural', 'Generated'])
        axes[i].set_title(name)
        axes[i].set_ylabel('Value')

    for j in range(len(feature_names), 16):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(output_dir + 'TDA_features_comparison.png')
    plt.show()

# Assume data['label'] contains the text labels (natural/generated)
labels = data['label'].values

plot_feature_boxplots(labels, features, ripser_feature_names)

In [ ]:


def plot_average_persistence_histogram(labels, layer=11, head=11):
    """
    Plot overlapping 1D histograms of H0 persistence lengths (death - birth)
    for natural vs generated text, for a given layer and head.
    """
    barcode_files = sorted(glob.glob(output_dir + 'barcodes/*_part*of*.json'))
    all_barcodes = []
    for f in barcode_files:
        with open(f, 'r') as infile:
            bc_data = json.load(infile)
            layer_str = str(layer)
            head_str = str(head)
            if layer_str in bc_data and head_str in bc_data[layer_str]:
                all_barcodes.extend(bc_data[layer_str][head_str])

    print(f"Loaded {len(all_barcodes)} barcodes for Layer {layer}, Head {head}")
    if len(all_barcodes) == 0:
        return []

    nat_lengths, gen_lengths = [], []
    for lbl, bc in zip(labels, all_barcodes):
        if '0' in bc:
            for pt in bc['0']:
                birth, death = pt[0], pt[1]
                if death != float('inf') and death < 1e5:
                    length = death - birth
                    if length > 0:
                        if lbl == 'natural':
                            nat_lengths.append(length)
                        else:
                            gen_lengths.append(length)

    nat_lengths = np.array(nat_lengths)
    gen_lengths = np.array(gen_lengths)

    fig, ax = plt.subplots(figsize=(12, 6))

    # Determine shared bin edges
    all_lengths = np.concatenate([nat_lengths, gen_lengths])
    bins = np.linspace(all_lengths.min(), np.percentile(all_lengths, 99), 60)

    ax.hist(nat_lengths, bins=bins, alpha=0.55, color='steelblue',
            label=f'Natural (n={len(nat_lengths)})', density=True, edgecolor='none')
    ax.hist(gen_lengths, bins=bins, alpha=0.55, color='tomato',
            label=f'Generated (n={len(gen_lengths)})', density=True, edgecolor='none')

    # Overlay KDE curves
    from scipy.stats import gaussian_kde
    if len(nat_lengths) > 1:
        kde_nat = gaussian_kde(nat_lengths)
        xs = np.linspace(bins[0], bins[-1], 300)
        ax.plot(xs, kde_nat(xs), color='steelblue', linewidth=2.5, linestyle='--')
    if len(gen_lengths) > 1:
        kde_gen = gaussian_kde(gen_lengths)
        xs = np.linspace(bins[0], bins[-1], 300)
        ax.plot(xs, kde_gen(xs), color='tomato', linewidth=2.5, linestyle='--')

    ax.set_xlabel('Persistence Length (death − birth)', fontsize=13)
    ax.set_ylabel('Density', fontsize=13)
    ax.set_title(f'H₀ Persistence Length Distribution — Layer {layer}, Head {head}', fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_dir + 'Average_Persistence_Histogram.png', dpi=150)
    plt.show()
    print(f"  Natural  mean persistence: {np.mean(nat_lengths):.4f} ± {np.std(nat_lengths):.4f}")
    print(f"  Generated mean persistence: {np.mean(gen_lengths):.4f} ± {np.std(gen_lengths):.4f}")
    return all_barcodes

barcodes_l11_h11 = plot_average_persistence_histogram(labels, layer=11, head=11)

In [ ]:


def calculate_bottleneck_distances(labels, barcodes, sample_size=30):
    print(f"Calculating bottleneck distances on a subset of {sample_size} samples...")
    nat_idx = np.where(labels == 'natural')[0]
    gen_idx = np.where(labels == 'generated')[0]

    np.random.seed(42)
    # Ensure we don't oversample beyond available data
    k_nat = min(sample_size, len(nat_idx))
    k_gen = min(sample_size, len(gen_idx))
    sample_nat_idx = np.random.choice(nat_idx, size=k_nat, replace=False)
    sample_gen_idx = np.random.choice(gen_idx, size=k_gen, replace=False)

    def get_h0(idx):
        bc = barcodes[idx]
        if '0' in bc:
            pts = [pt for pt in bc['0'] if pt[1] != float('inf') and pt[1] < 1e5]
            if len(pts) == 0:
                return np.empty((0, 2))
            return np.array(pts)
        return np.empty((0, 2))

    nat_diagrams = [get_h0(i) for i in sample_nat_idx]
    gen_diagrams = [get_h0(i) for i in sample_gen_idx]

    dist_nat_nat, dist_gen_gen, dist_nat_gen = [], [], []

    print("Computing Nat-Nat pairwise distances...")
    for i in tqdm(range(len(nat_diagrams))):
        for j in range(i+1, len(nat_diagrams)):
            dist_nat_nat.append(persim.bottleneck(nat_diagrams[i], nat_diagrams[j]))

    print("Computing Gen-Gen pairwise distances...")
    for i in tqdm(range(len(gen_diagrams))):
        for j in range(i+1, len(gen_diagrams)):
            dist_gen_gen.append(persim.bottleneck(gen_diagrams[i], gen_diagrams[j]))

    print("Computing Nat-Gen pairwise distances...")
    for i in tqdm(range(len(nat_diagrams))):
        for j in range(len(gen_diagrams)):
            dist_nat_gen.append(persim.bottleneck(nat_diagrams[i], gen_diagrams[j]))

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.boxplot([dist_nat_nat, dist_gen_gen, dist_nat_gen], labels=['Nat vs Nat', 'Gen vs Gen', 'Nat vs Gen'])
    ax.set_title(f"Bottleneck Distances (Sample Size: {sample_size} per class)")
    ax.set_ylabel("Distance")

    plt.tight_layout()
    plt.savefig(output_dir + 'Bottleneck_Distances.png')
    plt.show()

if len(barcodes_l11_h11) > 0:
    calculate_bottleneck_distances(labels, barcodes_l11_h11, sample_size=30)


# ## Takens' (Time-Delay) Embedding — BERT Layers as Timestamps
#
# For each text sample we treat the 12 BERT layers as a discrete time axis.
# At each layer we extract a scalar TDA feature (mean H0 persistence, averaged
# across all 12 heads).  This gives a 12-step time series per sample.
# Takens' theorem says: embed that scalar time series with delay τ and
# dimension m to reconstruct the attractor of the underlying dynamical system.
# We then average the reconstructed trajectories separately over natural and
# generated text samples and plot the results in 3-D.

In [ ]:


import matplotlib
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 – registers 3d projection

def takens_embedding(signal, m=3, tau=1):
    """
    Return the Takens (time-delay) embedding of a 1-D signal.

    Parameters
    ----------
    signal : array-like, shape (T,)
    m      : int   – embedding dimension
    tau    : int   – time delay

    Returns
    -------
    embedded : np.ndarray, shape (T - (m-1)*tau, m)
    """
    signal = np.asarray(signal, dtype=float)
    T = len(signal)
    n_points = T - (m - 1) * tau
    if n_points <= 0:
        raise ValueError(f"Signal length {T} too short for m={m}, tau={tau}")
    embedded = np.empty((n_points, m))
    for i in range(n_points):
        for j in range(m):
            embedded[i, j] = signal[i + j * tau]
    return embedded


def takens_layer_analysis(labels, ripser_file, output_dir, tau=1, m=3, feature_idx=0):
    """
    Main Takens embedding analysis.

    *feature_idx* selects which ripser feature is used as the scalar at each
    layer.  Default 0 → h0_s (sum of H0 persistence lengths).
    """
    print("\n=== Takens' Embedding Analysis ===")
    ripser_features = np.load(ripser_file)          # (12, 12, n_samples, n_feat)
    n_layers, n_heads, n_samples, n_feat = ripser_features.shape
    labels_used = labels[:n_samples]

    # Average across heads → (12, n_samples)
    layer_signals_all_heads = np.mean(ripser_features[:, :, :, feature_idx], axis=1)
    # Transpose → (n_samples, 12)
    X_ts = layer_signals_all_heads.T

    # Normalise per sample (z-score) to make trajectories comparable
    mu  = X_ts.mean(axis=1, keepdims=True)
    sig = X_ts.std(axis=1, keepdims=True) + 1e-9
    X_ts_norm = (X_ts - mu) / sig

    nat_mask = labels_used == 'natural'
    gen_mask = labels_used == 'generated'

    # Embed each sample and collect
    nat_embedded, gen_embedded = [], []
    for i in range(n_samples):
        try:
            emb = takens_embedding(X_ts_norm[i], m=m, tau=tau)
            if labels_used[i] == 'natural':
                nat_embedded.append(emb)
            else:
                gen_embedded.append(emb)
        except ValueError:
            pass

    # Average trajectories (point-wise in embedding space)
    nat_traj = np.mean(nat_embedded, axis=0)   # (n_points, m)
    gen_traj = np.mean(gen_embedded, axis=0)

    print(f"  Embedding dim m={m}, delay τ={tau}")
    print(f"  Natural samples: {len(nat_embedded)}, Generated: {len(gen_embedded)}")
    print(f"  Trajectory length: {nat_traj.shape[0]} points in {m}-D space")

    # ── 3-D trajectory plot ──────────────────────────────────────────────────
    fig = plt.figure(figsize=(13, 8))
    ax  = fig.add_subplot(111, projection='3d')

    def _plot_traj(ax, traj, color, label):
        n = traj.shape[0]
        # gradient line: opacity increases along trajectory to show direction
        for k in range(n - 1):
            alpha = 0.35 + 0.65 * k / max(n - 2, 1)
            ax.plot(traj[k:k+2, 0], traj[k:k+2, 1], traj[k:k+2, 2],
                    color=color, alpha=alpha, linewidth=2.5)
        ax.scatter(*traj[0],  color=color, s=120, marker='^',
                   zorder=5, label=f'{label} – start')
        ax.scatter(*traj[-1], color=color, s=120, marker='s',
                   zorder=5, label=f'{label} – end')

    _plot_traj(ax, nat_traj, '#2563EB', 'Natural')
    _plot_traj(ax, gen_traj, '#DC2626', 'Generated')

    ax.set_xlabel(f'Layer t (dim 1)', fontsize=11)
    ax.set_ylabel(f'Layer t+{tau} (dim 2)', fontsize=11)
    ax.set_zlabel(f'Layer t+{2*tau} (dim 3)', fontsize=11)
    ax.set_title(
        f"Takens' Embedding — Average Trajectory (τ={tau}, m={m})\n"
        f"Feature: {ripser_feature_names[feature_idx]} | Averaged across BERT heads",
        fontsize=12
    )
    ax.legend(fontsize=10, loc='upper left')
    fig.tight_layout()
    fig.savefig(output_dir + 'Takens_Embedding_3D.png', dpi=150)
    plt.show()

    # ── 2-D projections for each pair of dimensions ──────────────────────────
    dim_pairs = [(0, 1), (0, 2), (1, 2)]
    dim_labels = [
        (f'Layer t', f'Layer t+{tau}'),
        (f'Layer t', f'Layer t+{2*tau}'),
        (f'Layer t+{tau}', f'Layer t+{2*tau}'),
    ]
    fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
    for ax2, (d1, d2), (xl, yl) in zip(axes2, dim_pairs, dim_labels):
        ax2.plot(nat_traj[:, d1], nat_traj[:, d2], 'o-', color='#2563EB',
                 linewidth=2, label='Natural')
        ax2.plot(gen_traj[:, d1], gen_traj[:, d2], 'o-', color='#DC2626',
                 linewidth=2, label='Generated')
        # annotate time steps
        for k, (nx, ny) in enumerate(zip(nat_traj[:, d1], nat_traj[:, d2])):
            ax2.annotate(str(k), (nx, ny), fontsize=7, color='#2563EB', alpha=0.7)
        ax2.set_xlabel(xl, fontsize=11)
        ax2.set_ylabel(yl, fontsize=11)
        ax2.set_title(f'Projection ({xl} vs {yl})', fontsize=11)
        ax2.legend(fontsize=10)
        ax2.grid(alpha=0.3)
    fig2.suptitle(
        f"Takens' Embedding — 2-D Projections (τ={tau}, m={m})",
        fontsize=13, y=1.02
    )
    fig2.tight_layout()
    fig2.savefig(output_dir + 'Takens_Embedding_2D_Projections.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Per-layer mean TDA scalar (the raw time series before embedding) ─────
    fig3, ax3 = plt.subplots(figsize=(11, 5))
    layer_idx = np.arange(n_layers)
    nat_mean_ts = X_ts_norm[nat_mask].mean(axis=0)
    gen_mean_ts = X_ts_norm[gen_mask].mean(axis=0)
    nat_std_ts  = X_ts_norm[nat_mask].std(axis=0)
    gen_std_ts  = X_ts_norm[gen_mask].std(axis=0)

    ax3.plot(layer_idx, nat_mean_ts, 'o-', color='#2563EB', linewidth=2, label='Natural')
    ax3.fill_between(layer_idx, nat_mean_ts - nat_std_ts, nat_mean_ts + nat_std_ts,
                     color='#2563EB', alpha=0.15)
    ax3.plot(layer_idx, gen_mean_ts, 'o-', color='#DC2626', linewidth=2, label='Generated')
    ax3.fill_between(layer_idx, gen_mean_ts - gen_std_ts, gen_mean_ts + gen_std_ts,
                     color='#DC2626', alpha=0.15)
    ax3.set_xticks(layer_idx)
    ax3.set_xticklabels([f'L{i}' for i in layer_idx])
    ax3.set_xlabel('BERT Layer', fontsize=12)
    ax3.set_ylabel(f'Normalised {ripser_feature_names[feature_idx]} (mean ± 1σ)', fontsize=12)
    ax3.set_title('TDA Signal Across BERT Layers (pre-embedding time series)', fontsize=13)
    ax3.legend(fontsize=11)
    ax3.grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(output_dir + 'Takens_Layer_Signal.png', dpi=150)
    plt.show()

    return nat_traj, gen_traj, nat_embedded, gen_embedded


nat_traj, gen_traj, nat_emb, gen_emb = takens_layer_analysis(
    labels, ripser_file, output_dir, tau=1, m=3, feature_idx=0
)


# ## Mapper Algorithm
#
# The Mapper algorithm (Singh, Mémoli, Carlsson 2007) builds a topological
# summary graph of a point cloud.  Here we:
#   1. Build a feature matrix: one row per sample, columns = TDA ripser features
#      averaged across all layers and heads.
#   2. Use a 2-D PCA projection as the *lens* (filter function).
#   3. Cover the lens range with overlapping intervals; cluster each pre-image
#      with DBSCAN; connect clusters that share points.
#   4. Visualise the resulting graph both as an interactive HTML file (KeplerMapper)
#      and as a static matplotlib plot with nodes coloured by class proportion.

In [ ]:


def mapper_analysis(labels, ripser_file, output_dir):
    print("\n=== Mapper Algorithm Analysis ===")
    try:
        import kmapper as km
        import sklearn.cluster
        from sklearn.preprocessing import StandardScaler
        from sklearn.decomposition import PCA
        import networkx as nx
    except ImportError as e:
        print(f"  Missing dependency: {e}")
        print("  Install with: pip install kmapper networkx scikit-learn")
        return None, None

    ripser_features = np.load(ripser_file)       # (12, 12, n_samples, n_feat)
    n_layers, n_heads, n_samples, n_feat = ripser_features.shape
    labels_used = labels[:n_samples]

    # ── Build feature matrix ─────────────────────────────────────────────────
    # Average across layers and heads → (n_samples, n_feat)
    X_raw = np.mean(ripser_features, axis=(0, 1))   # (n_samples, n_feat)
    X_raw = np.nan_to_num(X_raw, nan=0.0, posinf=0.0, neginf=0.0)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)

    # Class vector: 1 = natural, 0 = generated
    nat_vec = (labels_used == 'natural').astype(float)

    # ── Lens: 2-D PCA projection ─────────────────────────────────────────────
    pca = PCA(n_components=2)
    lens = pca.fit_transform(X)
    print(f"  PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")

    # ── Mapper graph ─────────────────────────────────────────────────────────
    mapper = km.KeplerMapper(verbose=0)
    graph  = mapper.map(
        lens,
        X,
        cover=km.Cover(n_cubes=12, perc_overlap=0.5),
        clusterer=sklearn.cluster.DBSCAN(eps=0.8, min_samples=3)
    )

    n_nodes = len(graph['nodes'])
    n_edges = sum(len(v) for v in graph['links'].values())
    print(f"  Mapper graph: {n_nodes} nodes, {n_edges} edges")

    # ── Interactive HTML output ───────────────────────────────────────────────
    html = mapper.visualize(
        graph,
        title="Mapper Graph — TDA Features (BERT Attention)",
        custom_tooltips=labels_used,
        color_values=nat_vec,
        color_function_name="Fraction Natural",
    )
    html_path = output_dir + 'Mapper_graph.html'
    with open(html_path, 'w') as fh:
        fh.write(html)
    print(f"  Interactive HTML saved → {html_path}")

    # ── Static matplotlib graph ───────────────────────────────────────────────
    # Build networkx graph and colour nodes by natural fraction
    G = nx.Graph()
    for node in graph['nodes']:
        member_ids = graph['nodes'][node]
        nat_frac   = np.mean(nat_vec[member_ids])
        node_size  = len(member_ids)
        G.add_node(node, nat_frac=nat_frac, size=node_size)
    for src, dsts in graph['links'].items():
        for dst in dsts:
            G.add_edge(src, dst)

    if len(G.nodes) == 0:
        print("  Empty graph — try adjusting cover/clusterer parameters.")
        return graph, None

    pos        = nx.spring_layout(G, seed=42, k=2.0)
    nat_fracs  = np.array([G.nodes[n]['nat_frac'] for n in G.nodes])
    node_sizes = np.array([G.nodes[n]['size']     for n in G.nodes]) * 40 + 80

    cmap = plt.cm.RdYlBu
    fig, ax = plt.subplots(figsize=(14, 9))
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.35, edge_color='#94a3b8', width=1.5)
    sc = ax.scatter(
        [pos[n][0] for n in G.nodes],
        [pos[n][1] for n in G.nodes],
        c=nat_fracs, cmap=cmap, s=node_sizes,
        vmin=0, vmax=1, zorder=3, edgecolors='white', linewidths=0.8
    )
    for node in G.nodes:
        # label: size of cluster
        ax.annotate(
            str(G.nodes[node]['size']),
            pos[node], ha='center', va='center',
            fontsize=7, color='black', zorder=4
        )
    cbar = fig.colorbar(sc, ax=ax, shrink=0.7)
    cbar.set_label('Fraction Natural text', fontsize=11)
    ax.set_title(
        'Mapper Graph — TDA Feature Space\n'
        'Node colour: fraction of natural text samples  |  Node size: cluster population',
        fontsize=13
    )
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(output_dir + 'Mapper_graph_static.png', dpi=150)
    plt.show()

    # ── PCA scatter coloured by class (context for lens) ─────────────────────
    fig2, ax2 = plt.subplots(figsize=(9, 7))
    colors = ['#2563EB' if l == 'natural' else '#DC2626' for l in labels_used]
    ax2.scatter(lens[:, 0], lens[:, 1], c=colors, alpha=0.55, s=25, edgecolors='none')
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#2563EB', markersize=10, label='Natural'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#DC2626', markersize=10, label='Generated'),
    ]
    ax2.legend(handles=legend_elements, fontsize=11)
    ax2.set_xlabel(f'PC 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
    ax2.set_ylabel(f'PC 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
    ax2.set_title('PCA Lens — TDA Feature Space (Mapper input projection)', fontsize=13)
    ax2.grid(alpha=0.2)
    fig2.tight_layout()
    fig2.savefig(output_dir + 'Mapper_PCA_lens.png', dpi=150)
    plt.show()

    # ── Node-level statistics ─────────────────────────────────────────────────
    print("\n  Mapper node summary:")
    print(f"  {'Node':<30}  {'Size':>6}  {'Nat%':>6}  {'Gen%':>6}")
    for node in sorted(G.nodes, key=lambda n: G.nodes[n]['size'], reverse=True)[:15]:
        sz = G.nodes[node]['size']
        nf = G.nodes[node]['nat_frac']
        print(f"  {node:<30}  {sz:>6}  {nf*100:>5.1f}%  {(1-nf)*100:>5.1f}%")

    return graph, G


graph_mapper, G_mapper = mapper_analysis(labels, ripser_file, output_dir)